# 04b — Costruzione Feature Matrix

Produce `data/feature_matrix_graph_v1.parquet`:
- 9096 righe (agenti nel grafo conversazionale)
- 20 colonne: `agent_id` + 19 feature ML
- Zero NaN
- Feature skewed trasformate con log1p

**Prerequisiti**: 04a eseguito senza BUG.

## 1. Load dal DB

In [2]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../data/moltbook.db")

df_raw = pd.read_sql("SELECT * FROM agent_features", conn)
conn.close()

print(f"Righe totali: {len(df_raw)}")
print(f"Colonne: {list(df_raw.columns)}")

Righe totali: 27107
Colonne: ['agent_id', 'computed_at', 'feature_version', 'n_posts', 'n_comments', 'n_comments_received', 'first_activity', 'last_activity', 'active_days', 'burstiness_posts', 'hour_entropy', 'reply_to_post_ratio', 'self_reply_rate', 'unique_targets', 'mean_thread_depth', 'mean_post_length', 'std_post_length', 'type_token_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'local_clustering', 'egonet_size', 'egonet_density', 'reciprocity_local', 'is_claimed']


## 2. Filtro subset analitico

Teniamo solo gli agenti nel grafo conversazionale (in_degree valorizzato).

In [3]:
df_graph = df_raw[df_raw["in_degree"].notna()].copy()

print(f"Agenti nel grafo: {len(df_graph)}")
assert len(df_graph) >= 9000, f"Attesi almeno 9000 agenti, trovati {len(df_graph)} — verificare dataset"
print(f"Shape: {df_graph.shape}")

Agenti nel grafo: 9096
Shape: (9096, 27)


## 3. Selezione ML features

19 feature selezionate. Escluse:
- `unique_targets`: correlazione ~1.0 con `out_degree`
- `egonet_size`: correlazione ~1.0 con `out_degree`
- `is_claimed`: 98.9% claimed → sbilanciamento estremo, non utile per clustering
- Colonne di metadati: `computed_at`, `feature_version`, `first_activity`, `last_activity`

In [4]:
ML_FEATURES = [
    # Attività
    "n_posts", "n_comments", "n_comments_received", "active_days",
    # Temporali
    "burstiness_posts", "hour_entropy",
    # Comportamentali
    "reply_to_post_ratio", "self_reply_rate", "mean_thread_depth",
    # Testuali
    "mean_post_length", "std_post_length", "type_token_ratio",
    # Rete
    "in_degree", "out_degree", "pagerank", "betweenness",
    "local_clustering", "egonet_density", "reciprocity_local",
]
# NOTA: unique_targets e egonet_size esclusi per ridondanza (corr > 0.9999 con out_degree)

df_ml = df_graph[["agent_id"] + ML_FEATURES].copy()

print(f"Shape dopo selezione: {df_ml.shape}")
print(f"\nNaN per feature (prima dell'imputazione):")
nan_before = df_ml[ML_FEATURES].isnull().sum()
print(nan_before[nan_before > 0].to_string())

Shape dopo selezione: (9096, 20)

NaN per feature (prima dell'imputazione):
burstiness_posts    5082
self_reply_rate     4054
mean_post_length    2466
std_post_length     2466


## 4. Policy NaN — imputazione con 0

Tutti i NaN nelle feature ML sono strutturalmente giustificati (verificato in 04a).
L'imputazione con 0 è semanticamente corretta per ogni caso:

| Feature | Causa NaN | Imputazione con 0 significa |
|---|---|---|
| `burstiness_posts` | n_posts < 3 | nessuna burstiness misurabile |
| `self_reply_rate` | nessun commento con parent_id | non si è mai risposto a se stesso |
| `mean_post_length` | n_posts = 0 | agente senza post |
| `std_post_length` | n_posts = 0 | agente senza post |
| `mean_thread_depth` | depth NULL nel DB | profondità non misurabile |
| `reply_to_post_ratio` | n_posts=0 e n_comments=0 | nessuna attività |
| `type_token_ratio` | nessun testo | vocabolario non misurabile |

In [5]:
df_ml[ML_FEATURES] = df_ml[ML_FEATURES].fillna(0)

nan_after = df_ml[ML_FEATURES].isnull().sum().sum()
print(f"NaN totali dopo imputazione: {nan_after}")
assert nan_after == 0, "Ancora NaN dopo fillna — verificare"
print("OK — zero NaN")

NaN totali dopo imputazione: 0
OK — zero NaN


## 5. Log-transform sulle feature skewed

Trasformiamo le feature con distribuzione fortemente skewed.

- **log1p** (`log(1+x)`, sicuro per x=0): feature di conteggio e lunghezza testo
- **log10** (con epsilon=1e-6 come floor per gli zeri): feature di centralità (pagerank, betweenness) — distribuite su ordini di grandezza diversi, log10 è più interpretabile

NON trasformiamo:
- Feature già bounded in [0,1]: `hour_entropy`, `type_token_ratio`, `local_clustering`,
  `self_reply_rate`, `egonet_density`, `reciprocity_local`, `reply_to_post_ratio`
- Feature che possono essere negative: `burstiness_posts` (range [-1, 1])
- Feature ordinali/conteggi piccoli: `active_days`, `mean_thread_depth`

In [6]:
SKEWED_FEATURES = [
    "n_posts", "n_comments", "n_comments_received",
    "in_degree", "out_degree",
    "mean_post_length", "std_post_length",
]

# log10 per feature di centralità (distribuite su ordini di grandezza)
# epsilon=1e-6 come floor per gli zeri (betweenness può essere 0)
LOG10_FEATURES = ["pagerank", "betweenness"]
LOG10_EPSILON = 1e-6

# --- log1p ---
for col in SKEWED_FEATURES:
    min_val = df_ml[col].min()
    if min_val < 0:
        print(f"ATTENZIONE: {col} ha valori negativi (min={min_val:.4f}) — skip log1p")
    else:
        df_ml[col] = np.log1p(df_ml[col])

print("log1p applicato a:", SKEWED_FEATURES)

# --- log10 ---
for col in LOG10_FEATURES:
    min_val = df_ml[col].min()
    if min_val < 0:
        print(f"ATTENZIONE: {col} ha valori negativi (min={min_val:.6f}) — skip log10")
    else:
        df_ml[col] = np.log10(df_ml[col] + LOG10_EPSILON)

print("log10 applicato a:", LOG10_FEATURES)

all_transformed = SKEWED_FEATURES + LOG10_FEATURES
print("\nRange post-trasformazione:")
print(df_ml[all_transformed].agg(["min", "max", "mean"]).round(4).to_string())

log1p applicato a: ['n_posts', 'n_comments', 'n_comments_received', 'in_degree', 'out_degree', 'mean_post_length', 'std_post_length']
log10 applicato a: ['pagerank', 'betweenness']

Range post-trasformazione:
      n_posts  n_comments  n_comments_received  in_degree  out_degree  mean_post_length  std_post_length  pagerank  betweenness
min    0.0000      0.6931               0.0000     0.0000      0.0000            0.0000           0.0000   -4.2655      -6.0000
max    8.9335      9.5002               8.8742     6.7417      6.6771           10.5967           9.6711   -2.0651      -1.5525
mean   1.4620      2.7965               1.5683     1.2385      0.9498            4.9434           3.0395   -4.0776      -5.5231


## 6. Verifiche finali

In [7]:
# Zero NaN
assert df_ml.isnull().sum().sum() == 0, "NaN residui nella feature matrix!"
print("NaN check: OK")

# Shape attesa
expected_cols = len(ML_FEATURES) + 1  # +1 per agent_id
assert df_ml.shape[1] == expected_cols, f"Colonne attese: {expected_cols}, trovate: {df_ml.shape[1]}"
print(f"Shape check: {df_ml.shape} — OK")

# Nessun valore infinito
inf_count = np.isinf(df_ml[ML_FEATURES]).sum().sum()
assert inf_count == 0, f"Trovati {inf_count} valori infiniti!"
print("Inf check: OK")

print("\nStatistiche descrittive post-trasformazione:")
print(df_ml[ML_FEATURES].describe().round(4).to_string())

NaN check: OK
Shape check: (9096, 20) — OK
Inf check: OK

Statistiche descrittive post-trasformazione:
         n_posts  n_comments  n_comments_received  active_days  burstiness_posts  hour_entropy  reply_to_post_ratio  self_reply_rate  mean_thread_depth  mean_post_length  std_post_length  type_token_ratio  in_degree  out_degree   pagerank  betweenness  local_clustering  egonet_density  reciprocity_local
count  9096.0000   9096.0000            9096.0000    9096.0000         9096.0000     9096.0000            9096.0000        9096.0000          9096.0000         9096.0000        9096.0000         9096.0000  9096.0000   9096.0000  9096.0000    9096.0000         9096.0000       9096.0000          9096.0000
mean      1.4620      2.7965               1.5683       7.3493            0.0479        0.5251               0.7718           0.0140             0.3405            4.9434           3.0395            0.4751     1.2385      0.9498    -4.0776      -5.5231            0.0980          0.2421  

In [8]:
print("\nHead (5 righe):")
print(df_ml.head().to_string())


Head (5 righe):
                                agent_id   n_posts  n_comments  n_comments_received  active_days  burstiness_posts  hour_entropy  reply_to_post_ratio  self_reply_rate  mean_thread_depth  mean_post_length  std_post_length  type_token_ratio  in_degree  out_degree  pagerank  betweenness  local_clustering  egonet_density  reciprocity_local
3   e12f789d-bf76-4f3d-a01a-a533603ba241  0.000000    2.079442             0.693147            2          0.000000      0.487665             1.000000              0.0                0.0          0.000000         0.000000          0.655052   0.693147    0.000000 -4.256245    -6.000000               0.0             0.0               0.00
4   620a975a-77b9-4cea-b53d-1d55c9d3d2a6  2.302585    1.609438             0.693147            5          0.228674      0.525981             0.307692              0.0                1.0          7.616448         7.027709          0.375809   0.693147    1.609438 -4.177373    -5.898054               0.0     

## 7. Salvataggio

In [9]:
output_path = "../data/feature_matrix_graph_v1.parquet"
df_ml.to_parquet(output_path, index=False)

print(f"Salvato: {output_path}")
print(f"Shape: {df_ml.shape}")
print(f"Colonne: {list(df_ml.columns)}")

# Verifica idempotenza: rileggi e controlla shape
df_check = pd.read_parquet(output_path)
assert df_check.shape == df_ml.shape, "Shape mismatch al ricaricamento!"
print("\nVerifica ricaricamento: OK")

Salvato: ../data/feature_matrix_graph_v1.parquet
Shape: (9096, 20)
Colonne: ['agent_id', 'n_posts', 'n_comments', 'n_comments_received', 'active_days', 'burstiness_posts', 'hour_entropy', 'reply_to_post_ratio', 'self_reply_rate', 'mean_thread_depth', 'mean_post_length', 'std_post_length', 'type_token_ratio', 'in_degree', 'out_degree', 'pagerank', 'betweenness', 'local_clustering', 'egonet_density', 'reciprocity_local']

Verifica ricaricamento: OK
